# World Rupture — Remotion render on Colab (T4)
Run top to bottom. **Runtime → Change runtime type → T4 GPU.**

Colab's NVIDIA driver is *compute-only*. Chrome needs the NVIDIA **graphics/EGL** libraries to use the GPU for WebGL, so Cell 2 installs the `libnvidia-gl` package matching your driver, and Cell 4 **verifies** whether Chrome really reports the T4 (not SwiftShader/CPU). If it does not, Cell 4 says so plainly instead of silently rendering on CPU.

In [ ]:
!nvidia-smi
!nproc
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/world_rupture'
except Exception as exc:
    OUTPUT_ROOT = '/content/out_wr'
    print('Drive not mounted:', exc)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('Outputs ->', OUTPUT_ROOT)

## 2) Node 22, ffmpeg, Chrome libs, Xvfb, and NVIDIA graphics (EGL/Vulkan) libs matching the driver

In [ ]:
import subprocess
def sh(c, check=False):
    r = subprocess.run(c, shell=True, text=True, capture_output=True)
    print(r.stdout[-1500:], r.stderr[-800:])
    if check and r.returncode:
        raise RuntimeError(c)
    return r

sh('curl -fsSL https://deb.nodesource.com/setup_22.x | bash -')
sh('apt-get update -qq')
sh('apt-get install -y nodejs ffmpeg xvfb libnss3 libdbus-1-3 libatk1.0-0 libgbm-dev libasound2 libxrandr2 libxkbcommon-dev libxfixes3 libxcomposite1 libxdamage1 libatk-bridge2.0-0 libpango-1.0-0 libcairo2 libcups2 libgtk-3-0 libx11-xcb1 libegl1 libgles2 libgl1 libglvnd0 libglx0 libvulkan1 vulkan-tools mesa-utils', check=True)

ver = subprocess.run('nvidia-smi --query-gpu=driver_version --format=csv,noheader', shell=True, text=True, capture_output=True).stdout.strip()
major = ver.split('.')[0] if ver else ''
print('NVIDIA driver:', ver)
installed = False
for pkg in [f'libnvidia-gl-{major}', f'libnvidia-gl-{major}-server']:
    r = sh(f'apt-get install -y --no-install-recommends {pkg}')
    if r.returncode == 0:
        print('installed', pkg)
        installed = True
        break
if not installed:
    print('WARNING: could not install libnvidia-gl for driver', ver, '- Chrome will likely fall back to SwiftShader (CPU).')
sh('ldconfig')
sh('ls /usr/lib/x86_64-linux-gnu | grep -i nvidia | head -30')
sh('node -v; npm -v; ffmpeg -version | head -1')

## 3) Get the project and install packages

In [ ]:
%cd /content
!rm -rf /content/AIVideo
!git clone https://github.com/abdullahmehmood1/AIVideo.git /content/AIVideo
%cd /content/AIVideo
!npm ci || npm install
!npx remotion browser ensure --chrome-mode=chrome-for-testing
!npx remotion versions
!npx remotion compositions src/index.ts

## 4) GPU VERIFICATION + 60-frame benchmark
Runs Chrome for each GL mode under a virtual display (Xvfb) and reads Chrome's reported GPU state. Picks the fastest mode that reports hardware acceleration; if none does, it tells you so and picks the fastest working mode.

In [ ]:
import os, re, subprocess, time
os.chdir('/content/AIVideo')
os.environ['DISPLAY'] = ':99'
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1920x1080x24'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

BASE = ['--chrome-mode=chrome-for-testing', '--enable-multiprocess-on-linux']
START = 0   # set to a frame inside a heavy 3D scene to benchmark it (e.g. 5200)
gpu_ok, any_ok = [], []
for gl in ['angle-egl', 'vulkan', 'egl', 'angle', 'swangle']:
    info = subprocess.run(['npx', 'remotion', 'gpu', f'--gl={gl}'] + BASE, capture_output=True, text=True)
    txt = info.stdout + info.stderr
    is_cpu = ('Hardware accelerated' not in txt) or bool(re.search(r'swiftshader|llvmpipe', txt, re.I))
    out = f'/content/bench_{gl}.mp4'
    t0 = time.time()
    r = subprocess.run(['npx', 'remotion', 'render', 'src/index.ts', 'Main', out, f'--frames={START}-{START+59}', '--codec=h264', '--muted', '--concurrency=2', f'--gl={gl}', '--overwrite'] + BASE, capture_output=True, text=True)
    dt = time.time() - t0
    ok = r.returncode == 0 and os.path.exists(out)
    print(f'{gl:10s} ok={ok} {"CPU" if is_cpu else "GPU"} {dt:6.1f}s/60f ({dt/60:.2f}s/frame)')
    print('    ', ' | '.join(l.strip() for l in txt.splitlines() if re.search(r'WebGL|Canvas|GPU|Vulkan', l))[:300])
    if ok:
        any_ok.append((dt, gl))
        if not is_cpu:
            gpu_ok.append((dt, gl))

if gpu_ok:
    BEST_GL = sorted(gpu_ok)[0][1]
    print('\nGPU WebGL CONFIRMED. Using', BEST_GL)
else:
    BEST_GL = sorted(any_ok)[0][1] if any_ok else 'swangle'
    print('\n*** Chrome did NOT report hardware WebGL on this runtime. Rendering will use the CPU (' + BEST_GL + '). ***')
    print('Try: Runtime -> Disconnect and delete runtime, reconnect to a fresh T4, re-run Cell 2. Or set PERF_MODE = "low" in src/config.ts.')

## 5) Quick preview (first 10 s)

In [ ]:
import os, subprocess
os.chdir('/content/AIVideo'); os.makedirs('out', exist_ok=True)
GL = globals().get('BEST_GL', 'angle-egl')
subprocess.run(['npx', 'remotion', 'render', 'src/index.ts', 'Main', 'out/preview.mp4', '--frames=0-299', '--codec=h264', '--crf=18', '--muted', '--concurrency=2', f'--gl={GL}', '--chrome-mode=chrome-for-testing', '--enable-multiprocess-on-linux', '--overwrite'], check=True)
try:
    from google.colab import files
    files.download('out/preview.mp4')
except Exception as e:
    print(e)

## 6) Full render in 1,800-frame chunks (resumable, saved to Drive)

In [ ]:
import os, shutil, subprocess, time
os.chdir('/content/AIVideo')
GL = globals().get('BEST_GL', 'angle-egl')
PARTS = os.path.join(OUTPUT_ROOT, 'parts'); os.makedirs(PARTS, exist_ok=True)
TOTAL, CHUNK = 19350, 1800
for i, s in enumerate(range(0, TOTAL, CHUNK)):
    e = min(s + CHUNK - 1, TOTAL - 1)
    tgt = os.path.join(PARTS, f'part_{i:02d}.mp4')
    if os.path.exists(tgt) and os.path.getsize(tgt) > 10000:
        print('skip', tgt); continue
    tmp = '/content/part_tmp.mp4'
    t0 = time.time()
    r = subprocess.run(['npx', 'remotion', 'render', 'src/index.ts', 'Main', tmp, f'--frames={s}-{e}', '--codec=h264', '--crf=18', '--muted', '--concurrency=2', f'--gl={GL}', '--chrome-mode=chrome-for-testing', '--enable-multiprocess-on-linux', '--overwrite'])
    if r.returncode:
        raise RuntimeError(f'chunk {i} failed')
    shutil.copy2(tmp, tgt)
    print('done', tgt, round((time.time() - t0) / 60, 1), 'min')

## 7) Join chunks + mux audio (if master_audio.wav exists)

In [ ]:
import glob, os, shutil, subprocess
parts = sorted(glob.glob(os.path.join(OUTPUT_ROOT, 'parts', 'part_*.mp4')))
lst = os.path.join(OUTPUT_ROOT, 'parts', 'list.txt')
open(lst, 'w').write(''.join(f"file '{p}'\n" for p in parts))
va = os.path.join(OUTPUT_ROOT, 'video_noaudio.mp4')
subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', lst, '-c', 'copy', va], check=True)
audio = os.path.join(OUTPUT_ROOT, 'master_audio.wav')
final = os.path.join(OUTPUT_ROOT, 'final.mp4')
if os.path.exists(audio):
    subprocess.run(['ffmpeg', '-y', '-i', va, '-i', audio, '-c:v', 'copy', '-c:a', 'aac', '-b:a', '320k', '-shortest', final], check=True)
else:
    shutil.copy2(va, final)
    print('No master_audio.wav yet; video-only final.')
print(final)